# Practice: Creating & Using Persistent Storage

For this practice, we are going to integrate persistent volumes (data storage) into our pod.
In the lab activity, you simply uploaded and downloaded files to and from the running pod.
This is using ephemeral storage, which is a data storage area that simply disappears when the container/pod shuts down.

| Component                                      | Points |
|------------------------------------------------|--------|
| Part 1: Persistent volume screenshot (pvoutput.png)       | 5      |
| Part 2: Data lives screenshot (datalives.png)              | 5      |
| Part 3: Copied data screenshot (copieddata.png)            | 10     |
| **Total**                                      | **20** |


##### Make sure you have read the information on [persistent volumes linked](https://kubernetes.io/docs/concepts/storage/persistent-volumes/) in the previous module.

# Part 0: Upload Files to K8S Jupyter Hub

If you are running this practice from the gp-engine JupyterHub and have not cloned your course repository to the gp-engine JupyterHub, you'll need to download all of the development files from the [Schedule](../module4_schedule.ipynb) page and upload them to the gp-engine JupyterHub instance.


# Part 1: Create persistent volume


## Modify the yml file


Navigate to  the `development_files` sub folder within module4 folder. Update `persistent_volume.yml` file. 

Specify the name of your persistent volume in line 4 (replace `your_name`) with your `{MU-SSO-ID}-pv`.


## Run the yml file


```BASH
kubectl -n gp-engine-mizzou-dsa-cloud create -f persistent_volume.yml
```

##### Progress Check: Did you see something similar to:

```BASH
jovyan@jupyter-sknnh-missouri-edu---62fbc201:~/module4/resources$ kubectl -n gp-engine-mizzou-dsa-cloud create -f persistent_volume.yml
persistentvolumeclaim/sknnh-pv created
```


## Check if the persistent volume is created


The command below will list all persistent volumes related to your namespace

```BASH
kubectl -n gp-engine-mizzou-dsa-cloud get pvc
```


### Upload a screen shot of your persistent volume

Capture a screen snip of the result of your persistent storage volume and save it as `pv_output.png` and upload it here ([module4/practices/](./) folder). For full points, please ensure the image is linked in the mark up below.

**(5 pts)**

![Your pv_output.png is MISSING](./pv_output.png)


## Part 2: Attach the persistent volume

The only way to access the persistent volume is by attaching it to a pod. This requires some modification of the yaml file of the pod to be set up.


## Modify pod yaml file to mount persistent volume

Open `pod_pvc.yml` in a text editor and perform the following:

- line 4: Update pod name to `pod-pvc-{MU-SSO-ID}`
- line 23: Add the persistent volume name as specified in the previous section.
- line 21: Specify a name of the mounted volume to be recognized by the pod, for simplicity you can use the same name as the persistent volume
- line 19: This needs to be the same as in line 21.
- line 18: This is the folder name that appears when you access your pod


### Run the yml file

##### Take note to update the commands below as appropriate for your names

```BASH
kubectl -n gp-engine-mizzou-dsa-cloud create -f pod_pvc.yml
```

#### Confirm the pod is running, then access it.

```
kubectl -n gp-engine-mizzou-dsa-cloud exec -it pod_name -- /bin/bash
```

#### Once you are in the pod, confirm the file storage is attached

We can use the `df` command to confirm that we have a `/data` folder with 50 GB of storage.
Note, that `/data` in this case would be the edit on line 18 of the `pod_pvc.yml`

```BASH
root@pod-name-sknnh:/# df --output=size,used,target,avail,pcent -h --type ceph
 Size  Used Mounted on Avail Use%
  50G     0 /data        50G   0%
```

#### Write some data into the `/data` folder

We will capture the CPU Information for our pod into a text file.
Note, `/data` is what I chose for line 18 of the `pod_pvc.yml`

```BASH
cat /proc/cpuinfo > /data/cpuinfo
```


#### Make another data directory and name it `ccdata`

```bash
root@pod-name-sknnh:/# mkdir ccdata

# varify
root@pod-name-sknnh:/# ls
bin  boot  ccdata  data  dev  etc  home  lib  lib32  lib64  libx32  media  mnt  opt  proc  root  run  sbin  srv  sys  tmp  usr  var
```

## Delete the Pod

Refer to the lab if necessary to:

- close connection to the pod
- delete the pod

## Re-spawn the pod, and check that your data is still there

Refer to the lab if necessary to:

- create the pod again
- access the pod

#### Test that you data is still there:

Run this command, as before the `/data` should be customized based on your choices.

```BASH
grep -c processor /data/cpuinfo
```

This tells you how many processing cores the pod believes it has!

Then run this command

```BASH
uptime
```


### Capture an artifact screen snip of the output from the commands above.

Name the file `data_lives.png` and upload into the `module5/practices/` and ensure it links and shows in this notebook for full points.

#### The screen shot should show you creating a new pod, accessing it, and then processing the existing data (not creating the data again).

**(5 pts)**

![YOUR data_lives.png IS MISSING](./data_lives.png)


### Delete the pod

After you have successfully capatured the necessary screenshot, refer to the lab if necessary to close your connection to the pod and delete it.


# Part 3: Copy data from one persistent volume to another


Kubernetes allows that to mount more than one persistent volume to one pod.
This grants us access to both persistent volumes at the same time provided that they are both a part of the same namespace.


We have prepared data for you to use in your exercise but you need to move it to the persistent volume you created.

A pod may not be enough for such task,
after all pods are meant for simple tasks such as testing and development.
Instead we will be using a **job**.


## Modify the yaml file for the job


Open `job_copy_data.yml` in a text editor and perform the following:

- line 4: replace **sso** with your MU SSO ID
- line 10: replace **sso** with your MU SSO ID
- line 33: Add the name of your persistent volume
- line 18: Replace _ccdata_ with the path where your PV will be mounted
- line 14: Replace _ccdata_ with the same path from line 18

**Note**: You may also keep lines 14 and 18 unchanged if your mount path is _ccdata_. This is actually the folder you created to store your data. 


## Run the job yml file

```BASH
kubectl -n gp-engine-mizzou-dsa-cloud create -f job_copy_data.yml
```


## Check jobs

```BASH
kubectl get jobs
```

#### Do you see a job with duration?

```BASH
jovyan@jupyter-sknnh-missouri-edu---62fbc201:~/module4/resources$ kubectl get jobs
NAME                      STATUS    COMPLETIONS   DURATION   AGE
job-data-download-sknnh   Running   0/1           40s        40s
```


## Check pod

`kubectl get pods`

#### Progress Check: Do you see a running data-download pod with your MU SSO ID?

```BASH
jovyan@jupyter-sknnh-missouri-edu---62fbc201:~$ kubectl get pods
NAME                                 READY   STATUS    RESTARTS   AGE
job-data-download-sknnh--626z2   1/1     Running   0          55s
```


#### Error Checking:

If you have an error status, you can use the command (replacing pod-name with the NAME from the command above).:

```
kubectl -n gp-engine-mizzou-dsa-cloud describe pod pod-name
```


### Check for data copy success

Once you see that the job pob is completed you can proceed.  
Example, note the `STATUS` is `Completed`.

```BASH
jovyan@jupyter-sknnh-missouri-edu---62fbc201:~/module4/resources$ kubectl get jobs
NAME                       STATUS     COMPLETIONS   DURATION   AGE
job-data-download-sknnh    Complete   1/1           20s        2m19s
```

- Spin up a pod that has the mount (look up to Part 1).
- Connect to the pod and look for data

```BASH
kubectl exec -it pod_name -- /bin/bash

# If you are already in data directory, due to setting mount path,
# Do not change directory
# Otherwise change directory
cd /data

# List files
ls
```


##### Expected Check Output is Similar to this

```BASH
jovyan@jupyter-sknnh-missouri-edu---62fbc201:~/module4/resources$ kubectl -n gp-engine-mizzou-dsa-cloud exec -it pod-name-sknnh -- /bin/bash
root@pod-name-sknnh:/# cd data
root@pod-name-sknnh:/data# ls
LICENSE  README.md  main.py  models  utils.py
root@pod-name-sknnh:/data# 
```


### Artifact: take screen snip of that captures your terminal showing the expected contents and upload it to the [practices](./) folder. Name the file: `copied_data.png` and ensure it renders in this notebook to earn points.

**(10 pts)**

#### <span style='background:yellow'>NOTE: This data is required to proceed with the Exercise!</span>


![YOUR copied_data.png is MISSING](./copied_data.png)


---

# Part 4: Clean Up!

Once you have captured all the necessary screen shots, please clean up **all** the pods and jobs.

**Important** Do not clean up your persistent volume storage, as the data you copied in is **required** for the Exercise!

Here are some hints on clean up, where you will need to update them a little. See the Kubernetes documentation.

- Cleaning up a job or pod: `kubectl -n gp-engine-mizzou-dsa-cloud delete -f some_job_file_name.yml`


# Save your notebook and ensure you are adding and commiting your artifacts!
